<a href="https://colab.research.google.com/github/cbonnin88/SoundStream/blob/main/Persona_Clusters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import polars as pl
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import gdown as gd

In [4]:
url = 'https://drive.google.com/uc?id=1TkZ3_vwEH6zCZAa6QZWzhtFhKreHinbk'
gd.download(url,'soundstream_user_behavior_cleaned.csv',quiet=True)

'soundstream_user_behavior_cleaned.csv'

In [7]:
df_soundstream = pl.read_csv('soundstream_user_behavior_cleaned.csv')

# **Segmentation (ML)**

In [9]:
# Select behavioral columns for clustering

features = ['avg_listening_hours_per_week','playlists_created','avg_skips_per_day']
X = df_soundstream.select(features).to_numpy()

In [11]:
# Standardize features

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [12]:
# Apply K-Means

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_soundstream = df_soundstream.with_columns(
    user_segment = pl.Series(kmeans.fit_predict(X_scaled)).cast(pl.Utf8)
)

In [14]:
# Visualizing the Segments
fig_ml = px.scatter_3d(
    df_soundstream.sample(5000).to_pandas(),
    x='avg_listening_hours_per_week',
    y='playlists_created',
    z='avg_skips_per_day',
    color='user_segment',
    title='3D User Behavioral Segmentation',
    opacity=0.7,
    labels = {'user_segment':'User Segments','avg_skips_per_day':'Avg Skips (Daily)','playlists_created':'Playlists Created','avg_listening_hours_per_week':'Avg Listening Hous (Weekly)'}

)

fig_ml.show()

# **Product Manager Summary of Clusters**

In [16]:
display(
    df_soundstream.group_by('user_segment').agg([
        pl.col('avg_listening_hours_per_week').mean().alias('avg_hours'),
        pl.col('playlists_created').mean().alias('avg_playlists'),
        pl.len().alias('segment_size')
    ]).sort('avg_hours')
)

user_segment,avg_hours,avg_playlists,segment_size
str,f64,f64,u32
"""0""",6.150047,6.818054,13537
"""1""",10.080398,7.377464,11262
"""3""",10.176787,11.552556,11892
"""2""",13.648533,6.564731,13309
